# Retrieval Evaluation Metrics — A Worked Tutorial

For **RAG retrieval evaluation**, the main question is: is your retriever returning the **right chunks** for a query? This notebook walks through every metric commonly used to answer that, each with a small worked-by-hand example, in pure Python — no LLM calls, no cost.

RAG evaluation as a whole splits into two branches — retrieval quality and generation quality — and this notebook covers only the left one:

```
                     RAG Evaluation
                          |
             +-------------------+-------------------+
             |                                        |
             v                                        v
        RETRIEVAL                               GENERATION
             |                                        |
    +--------+--------+                    +----------+----------+
    v        v        v                    v          v          v
 Recall   Precision  Hit Rate         Faithfulness  Relevancy  Correctness
    |        |        |                    |          |          |
    +--------+--------+                    +----------+----------+
             v                                        v
          Ranking                              Answer Quality
             |
       +-----+-----+
       v     v     v
      MRR   MAP   NDCG
```

| Metric | What it measures | Higher is |
|---|---|---|
| **Precision@K** | Of the top K chunks, how many are relevant? | Better |
| **Recall@K** | Of all relevant chunks, how many were retrieved in top K? | Better |
| **Hit Rate / Hit@K** | Did at least one relevant chunk appear in top K? | Better |
| **MRR** | How high is the first relevant chunk ranked? | Better |
| **NDCG@K** | How well are relevant chunks ranked, including graded relevance? | Better |
| **MAP@K** | Average precision across multiple relevant chunks | Better |
| **Context Precision** | Are the retrieved chunks relevant (LLM-judged, against a reference answer)? | Better |
| **Context Recall** | Did we retrieve everything the reference answer needed (LLM-judged)? | Better |

As the diagram above shows, this notebook covers the Retrieval branch only — the Generation branch (Faithfulness, Relevancy, Correctness) is a separate concern, covered by a future generation-quality-metrics notebook. The first six retrieval metrics are classic information-retrieval metrics: they assume you already know *which chunk IDs* are correct for a query. The last two are Ragas' retrieval metrics: instead of needing labeled chunk IDs, an LLM judges relevance/coverage against a reference answer — which is why they're the ones used on synthetic datasets like the one `l27_ragas_evaluation_dataset.ipynb` generates (no hand-labeled chunk IDs available), and applied for real in `l28_ragas_retrieval_metrics.ipynb`. This notebook shows the same underlying math with toy numbers so the formulas are transparent before trusting the LLM to fill them in.

## The running example

Suppose the correct (relevant) chunks for a question are `C2`, `C5`, `C8`. A retriever returns these top 5 chunks, in rank order:

In [1]:
relevant_chunks = {"C2", "C5", "C8"}
retrieved = ["C2", "C7", "C5", "C10", "C3"]  # rank 1 -> rank 5

for rank, chunk in enumerate(retrieved, start=1):
    mark = "\u2705" if chunk in relevant_chunks else "\u274c"
    print(f"rank {rank}: {chunk:>4}  {mark}")

rank 1:   C2  ✅
rank 2:   C7  ❌
rank 3:   C5  ✅
rank 4:  C10  ❌
rank 5:   C3  ❌


## 1. Precision@K

```
Precision@K = relevant retrieved / total retrieved (K)
```

Out of the top 5, `C2` and `C5` are relevant — 2 out of 5.

In [2]:
def precision_at_k(retrieved, relevant, k):
    top_k = retrieved[:k]
    relevant_retrieved = [c for c in top_k if c in relevant]
    return len(relevant_retrieved) / k

precision_at_k(retrieved, relevant_chunks, k=5)

0.4

## 2. Recall@K

```
Recall@K = relevant retrieved / total relevant
```

Of the 3 relevant chunks (`C2`, `C5`, `C8`), 2 showed up in the top 5 (`C8` was missed) — 2 out of 3.

In [3]:
def recall_at_k(retrieved, relevant, k):
    top_k = retrieved[:k]
    relevant_retrieved = [c for c in top_k if c in relevant]
    return len(relevant_retrieved) / len(relevant)

recall_at_k(retrieved, relevant_chunks, k=5)

0.6666666666666666

## 3. Hit Rate / Hit@K

The simplest metric: did **at least one** relevant chunk make it into the top K? It's binary (1 or 0) per query; averaged across many queries it becomes a "hit rate" (fraction of queries that got at least one relevant chunk).

In [4]:
def hit_at_k(retrieved, relevant, k):
    top_k = retrieved[:k]
    return 1 if any(c in relevant for c in top_k) else 0

print("Hit@5 (our example, C2 is relevant at rank 1):", hit_at_k(retrieved, relevant_chunks, k=5))

# contrast: a query where nothing relevant made the top K
missed_retrieval = ["C7", "C10", "C3", "C1", "C9"]
print("Hit@5 (a retrieval that misses entirely):", hit_at_k(missed_retrieval, relevant_chunks, k=5))

Hit@5 (our example, C2 is relevant at rank 1): 1
Hit@5 (a retrieval that misses entirely): 0


## 4. MRR (Mean Reciprocal Rank)

MRR focuses on **where the first correct result appears**, not how many relevant chunks were found:

```
RR (one query) = 1 / rank of first relevant chunk
MRR (many queries) = average of RR across all queries
```

If the first two ranks are misses and the third rank is a hit, `RR = 1/3 ≈ 0.33`. In our running example the very first chunk (`C2`) is relevant, so `RR = 1/1 = 1.0` — MRR rewards getting a relevant chunk to the top far more than getting several relevant chunks scattered lower down.

In [5]:
def reciprocal_rank(retrieved, relevant):
    for rank, chunk in enumerate(retrieved, start=1):
        if chunk in relevant:
            return 1 / rank
    return 0.0

print("RR (our example):", reciprocal_rank(retrieved, relevant_chunks))

# the doc's illustrative example: miss, miss, hit at rank 3
print("RR (miss, miss, hit at rank 3):", reciprocal_rank(["C7", "C10", "C2"], relevant_chunks))

# MRR is an average across queries -- a tiny toy "dataset" of 3 queries
queries_retrieved = [
    ["C2", "C7", "C5", "C10", "C3"],   # hit at rank 1
    ["C7", "C10", "C2", "C3", "C5"],   # hit at rank 3
    ["C7", "C10", "C3", "C1", "C9"],   # no hit at all
]
mrr = sum(reciprocal_rank(r, relevant_chunks) for r in queries_retrieved) / len(queries_retrieved)
print("MRR across 3 queries:", round(mrr, 3))

RR (our example): 1.0
RR (miss, miss, hit at rank 3): 0.3333333333333333
MRR across 3 queries: 0.444


## 5. NDCG@K (Normalized Discounted Cumulative Gain)

Precision/Recall/Hit/MRR all treat relevance as yes/no. NDCG is for when relevance is **graded** (e.g. 0 = irrelevant, 3 = highly relevant) and you care about the quality of the whole ranking, not just whether something relevant showed up.

```
DCG@K  = sum( relevance_i / log2(i + 1) )   for i = 1..K
IDCG@K = DCG@K of the ideal ranking (relevances sorted best-first)
NDCG@K = DCG@K / IDCG@K
```

Example: retrieved relevances by rank are `[3, 0, 1, 3]` (highly relevant, irrelevant, somewhat relevant, highly relevant).

In [6]:
import math

def dcg(relevances):
    return sum(rel / math.log2(i + 1) for i, rel in enumerate(relevances, start=1))

def ndcg(relevances):
    ideal = sorted(relevances, reverse=True)
    ideal_dcg = dcg(ideal)
    return dcg(relevances) / ideal_dcg if ideal_dcg > 0 else 0.0

graded_relevances = [3, 0, 1, 3]
print("DCG:", round(dcg(graded_relevances), 3))
print("Ideal ranking would be:", sorted(graded_relevances, reverse=True))
print("IDCG:", round(dcg(sorted(graded_relevances, reverse=True)), 3))
print("NDCG:", round(ndcg(graded_relevances), 3))

DCG: 4.792
Ideal ranking would be: [3, 3, 1, 0]
IDCG: 5.393
NDCG: 0.889


NDCG is well below 1.0 here because the two highly-relevant chunks (relevance 3) aren't both at the top — one of them is buried behind an irrelevant chunk. A ranking that put both `3`s first would score NDCG = 1.0 (it *is* the ideal ranking).

## 6. MAP@K (Mean Average Precision)

Average Precision (AP) for one query: walk down the ranked list, and every time you hit a relevant chunk, record Precision@(that rank). Average those precision values over the total number of relevant chunks (not just the ones you found).

In our running example: relevant chunks appear at rank 1 (`C2`) and rank 3 (`C5`):

```
P(1) = 1/1 = 1.00   (relevant hit at rank 1)
P(3) = 2/3 = 0.667  (relevant hit at rank 3)
AP@5 = (1.00 + 0.667) / 3 relevant chunks total = 0.556
```

(`C8`, the third relevant chunk, was never retrieved, so it contributes 0 to the sum but still counts in the denominator — that's what penalizes missed relevant chunks.) **MAP@K** is just the mean of AP@K across many queries, the same way MRR is the mean of RR.

In [6]:
def average_precision_at_k(retrieved, relevant, k):
    top_k = retrieved[:k]
    hits, precision_sum = 0, 0.0
    for rank, chunk in enumerate(top_k, start=1):
        if chunk in relevant:
            hits += 1
            precision_sum += hits / rank
    return precision_sum / len(relevant)

ap = average_precision_at_k(retrieved, relevant_chunks, k=5)
print("AP@5 (our example):", round(ap, 3))

# MAP@K across the same 3 toy queries used for MRR
map_at_5 = sum(average_precision_at_k(r, relevant_chunks, k=5) for r in queries_retrieved) / len(queries_retrieved)
print("MAP@5 across 3 queries:", round(map_at_5, 3))

AP@5 (our example): 0.556
MAP@5 across 3 queries: 0.267


## 7 & 8. Context Precision and Context Recall (Ragas)

Everything above needed ground-truth chunk **IDs** — you had to already know that `C2`, `C5`, `C8` were the correct chunks. Ragas' retrieval metrics solve a different problem: for a synthetic or hand-written dataset with just a **question** and a **reference answer** (no labeled chunk IDs at all), an LLM judges the retrieved chunks against that reference answer instead.

**Context Precision** turns out to be the *exact same Average-Precision formula as MAP@K above* — the only difference is where the binary "is this chunk relevant" verdict comes from. Above, relevance came from a known ID set (`chunk in relevant_chunks`); in Ragas, an LLM reads the chunk and the reference answer and outputs a verdict (1 = useful, 0 = not useful) for each retrieved chunk, in rank order. Plug those LLM verdicts into the same formula:

In [7]:
# Suppose an LLM judged these 5 retrieved chunks against a reference answer:
# verdict[i] = 1 if chunk i was useful for answering the question, else 0
llm_verdicts = [1, 0, 1, 0, 0]  # same pattern as our C2/C7/C5/C10/C3 example, but judged by an LLM

def context_precision(verdicts):
    hits, precision_sum = 0, 0.0
    total_relevant = sum(verdicts)
    if total_relevant == 0:
        return 0.0
    for rank, verdict in enumerate(verdicts, start=1):
        if verdict:
            hits += 1
            precision_sum += hits / rank
    return precision_sum / total_relevant

context_precision(llm_verdicts)

0.8333333333333333

**Context Recall** asks a different question: break the *reference answer* into individual claims/statements, and check how many of them can actually be traced back to the retrieved context.

```
Context Recall = claims in the reference answer supported by retrieved context / total claims in the reference answer
```

Example: the reference answer makes 4 factual claims, and the retrieved chunks support 3 of them:

In [8]:
reference_claims_supported = [True, True, False, True]  # judged by an LLM, one per claim in the reference answer

def context_recall(claims_supported):
    return sum(claims_supported) / len(claims_supported)

context_recall(reference_claims_supported)

0.75

In `l28_ragas_retrieval_metrics.ipynb`, `ragas.evaluate(..., metrics=[LLMContextPrecisionWithReference(), LLMContextRecall()])` does exactly this — it prompts an LLM to produce the verdicts/claim-support judgments above for every question in the dataset, then applies these same two formulas. Nothing about the math changes; the LLM just replaces the ground-truth ID lookup this notebook used for Precision@K/Recall@K/MRR/NDCG/MAP@K.

## Summary — picking metrics for a real project

```
                    Retrieval Evaluation
                           |
        +------------------+------------------+
        |                  |                  |
    Coverage             Ranking           Relevance
        |                  |                  |
   Recall@K              MRR@K           Precision@K
   Hit@K                 NDCG@K          Context Precision
   Context Recall        MAP@K
```

For a typical RAG project (especially with hybrid retrieval + reranking, as in `l26_retrieval_strategies.ipynb`), track at minimum:

1. **Recall@K** — did we retrieve the correct chunks at all?
2. **Precision@K** — how much of what we retrieved is actually relevant?
3. **MRR** — is the correct chunk near the top?
4. **NDCG@K** — is the overall ranking quality good (graded relevance)?
5. **Hit@K** — did we retrieve at least one correct chunk?

If you also rerank, evaluate the reranker on its own by comparing these metrics before vs. after it runs (e.g. Recall@20 out of the first-stage retriever vs. Recall@5 after reranking down to the final top 5) — that tells you whether poor RAG performance traces back to the retriever, the reranker, or both.

When you don't have hand-labeled relevant chunk IDs (e.g. a synthetic Ragas testset), reach for **Context Precision** / **Context Recall** instead — see `l28_ragas_retrieval_metrics.ipynb` for the applied version against the real `data/sampled_schemes.pdf` pipeline.